# 01 - Data Loading

**Dataset:** [Kaggle Fraud Detection](https://www.kaggle.com/datasets/kartik2112/fraud-detection)

Stage 1 of 4. This notebook loads the official Kaggle train/test split, runs the
data-integrity checks, builds the engineered feature matrix, and saves everything
into `output/` for the next three notebooks.

| Notebook | Stage |
|---|---|
| **01_data_loading.ipynb** | **Load + validate + feature engineering (this one)** |
| 02_eda.ipynb | Exploratory data analysis |
| 03_model_training.ipynb | Model training |
| 04_evaluation.ipynb | Evaluation and performance analysis |

## 1. Imports & setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
from pathlib import Path

KAGGLE_IN = Path("/kaggle/input")


def find_data_dir():
    """Folder holding fraudTrain.csv / fraudTest.csv (Kaggle input mount or local archive/)."""
    for c in [KAGGLE_IN / "fraud-detection", Path("archive"), Path(".")]:
        if (c / "fraudTrain.csv").exists():
            return c
    for p in sorted(KAGGLE_IN.rglob("fraudTrain.csv")) if KAGGLE_IN.exists() else []:
        return p.parent
    return Path("archive")


# Raw CSVs: Kaggle dataset mount when on Kaggle, else the local archive/ folder.
DATA_DIR = find_data_dir()

# Everything this project generates goes into output/.
OUT = Path("/kaggle/working/output") if Path("/kaggle/working").exists() else Path("output")
for sub in ["data", "plots", "models", "preds", "results"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)


def upstream(rel):
    """Locate a file written by an earlier notebook (local run or Kaggle kernel input)."""
    local = OUT / rel
    if local.exists():
        return local
    if KAGGLE_IN.exists():
        for p in sorted(KAGGLE_IN.rglob(rel.split("/")[-1])):
            if p.as_posix().endswith("output/" + rel):
                return p
    raise FileNotFoundError("Run the earlier notebook first - missing: " + rel)


def save_fig(name):
    """Save the current matplotlib figure into output/plots/."""
    plt.savefig(OUT / "plots" / (name + ".png"), dpi=150, bbox_inches="tight")


print("DATA_DIR:", DATA_DIR)
print("OUT     :", OUT)

## 2. Load data

Use the official Kaggle split: `fraudTrain.csv` for training, `fraudTest.csv` for testing (time-based split).

In [ ]:
train_raw = pd.read_csv(DATA_DIR / "fraudTrain.csv")
test_raw = pd.read_csv(DATA_DIR / "fraudTest.csv")

print("Train shape:", train_raw.shape)
print("Test shape :", test_raw.shape)
print("\nTrain class counts:\n", train_raw["is_fraud"].value_counts())
print("\nTrain fraud rate: {:.4f}%".format(100 * train_raw["is_fraud"].mean()))
print("Test  fraud rate: {:.4f}%".format(100 * test_raw["is_fraud"].mean()))
train_raw.head(3)

## 3. Dataset overview

Structure, dtypes, missing values, duplicates and the class distribution of the training file.

In [ ]:
df = train_raw

df.head()

In [ ]:
# Column names
print(df.columns)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df["is_fraud"].value_counts()

In [ ]:
df["is_fraud"].value_counts(normalize=True) * 100

In [ ]:
# Basic statistics
print(df.describe())

## 4. Data validation

Class distribution of both files and a train/test row-leakage check on `trans_num`.

In [ ]:
print("TRAIN rows:", len(train_raw), "| TEST rows:", len(test_raw))
print("\nTrain class counts:\n", train_raw["is_fraud"].value_counts())
print("Test  class counts:\n", test_raw["is_fraud"].value_counts())
print("\nTrain fraud rate: {:.4f}%".format(100 * train_raw["is_fraud"].mean()))
print("Test  fraud rate: {:.4f}%".format(100 * test_raw["is_fraud"].mean()))

overlap = set(train_raw["trans_num"]) & set(test_raw["trans_num"])
print("\nTrain/test overlapping trans_num:", len(overlap), "(expected: 0 - no row leakage)")

# Sample rows - fraud vs non-fraud look different in amount and category
print("\nSample FRAUD transactions (train):")
print(train_raw.loc[train_raw["is_fraud"] == 1, ["amt", "category", "gender"]].head(5).to_string(index=False))
print("\nSample NON-FRAUD transactions (train):")
print(train_raw.loc[train_raw["is_fraud"] == 0, ["amt", "category", "gender"]].head(5).to_string(index=False))

## 5. Feature engineering

We create useful signals and drop IDs / high-leakage text fields.

| New / kept feature | Why |
|---|---|
| `age` | Fraud patterns can differ by age group |
| `hour`, `dayofweek`, `is_night` | Time patterns (night fraud is common) |
| `distance_km` | Customer vs merchant location gap |
| `log_amt` | Amount is skewed; log helps linear models |
| `gender` | Encoded binary |
| `category_*` | One-hot of merchant category |
| `city_pop` | City size context |

Dropped: names, street, card number, transaction id, merchant name (very high cardinality).

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Distance between two lat/long points in kilometers."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["trans_date_trans_time"] = pd.to_datetime(out["trans_date_trans_time"])
    out["dob"] = pd.to_datetime(out["dob"])

    out["age"] = (out["trans_date_trans_time"] - out["dob"]).dt.days / 365.25
    out["hour"] = out["trans_date_trans_time"].dt.hour
    out["dayofweek"] = out["trans_date_trans_time"].dt.dayofweek
    out["is_night"] = out["hour"].between(0, 5).astype(int)

    out["distance_km"] = haversine_km(
        out["lat"].values, out["long"].values,
        out["merch_lat"].values, out["merch_long"].values,
    )
    out["log_amt"] = np.log1p(out["amt"])
    out["gender"] = (out["gender"] == "M").astype(int)

    keep = [
        "amt", "log_amt", "gender", "city_pop", "age",
        "hour", "dayofweek", "is_night", "distance_km",
        "category", "is_fraud",
    ]
    out = out[keep]

    # One-hot category (same columns later aligned for train/test)
    out = pd.get_dummies(out, columns=["category"], drop_first=True)
    return out

In [ ]:
train_fe = engineer_features(train_raw)
test_fe = engineer_features(test_raw)

# Align columns so train/test have the same features
train_fe, test_fe = train_fe.align(test_fe, join="outer", axis=1, fill_value=0)

# Keep target last
feature_cols = [c for c in train_fe.columns if c != "is_fraud"]
train_fe = train_fe[feature_cols + ["is_fraud"]]
test_fe = test_fe[feature_cols + ["is_fraud"]]

print("Engineered train shape:", train_fe.shape)
print("Engineered test shape :", test_fe.shape)
print("\nFeature list:")
print(feature_cols)
train_fe.head(3)

## 6. Save artifacts for the next notebooks

| File | Used by |
|---|---|
| `output/data/train_fe.parquet` | 02 (feature plots), 03 (training) |
| `output/data/test_fe.parquet` | 03 (test matrix) |
| `output/data/train_eda.parquet` | 02 (raw-column EDA plots) |
| `output/results/data_validation.csv` | reference |

In [ ]:
train_fe.to_parquet(OUT / "data" / "train_fe.parquet", index=False)
test_fe.to_parquet(OUT / "data" / "test_fe.parquet", index=False)

# Raw columns the EDA notebook plots from
eda_cols = ["is_fraud", "amt", "category", "trans_date_trans_time"]
train_raw[eda_cols].to_parquet(OUT / "data" / "train_eda.parquet", index=False)

validation = pd.DataFrame([
    {"split": "train", "rows": len(train_raw), "fraud": int(train_raw["is_fraud"].sum()),
     "fraud_rate_pct": 100 * train_raw["is_fraud"].mean()},
    {"split": "test", "rows": len(test_raw), "fraud": int(test_raw["is_fraud"].sum()),
     "fraud_rate_pct": 100 * test_raw["is_fraud"].mean()},
])
validation["trans_num_overlap"] = len(overlap)
validation.to_csv(OUT / "results" / "data_validation.csv", index=False)

print(validation.to_string(index=False))
print("\nSaved to", OUT)
for p in sorted((OUT / "data").iterdir()):
    print(" ", p.name, "{:.1f} MB".format(p.stat().st_size / 1e6))